<a href="https://colab.research.google.com/github/SREENIDHIPAGIDIMARRI/ML/blob/main/Project_ML.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


In [ ]:
!pip -q install kaggle opencv-python-headless faiss-cpu grad-cam scikit-learn seaborn reportlab

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.5/44.5 kB 1.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 87.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 69.3 MB/s eta 0:00:00


In [4]:
from google.colab import files

uploaded = files.upload()

Saving kaggle.json to kaggle.json


In [5]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [6]:
!kaggle datasets list -s "mobile phone defect segmentation"

ref                                                   title                                           size  lastUpdated                 downloadCount  voteCount  usabilityRating  
----------------------------------------------------  ----------------------------------------  ----------  --------------------------  -------------  ---------  ---------------  
girish17019/mobile-phone-defect-segmentation-dataset  Mobile Phone Defect Segmentation Dataset  1723712627  2023-06-21 07:33:51.957000           1999         25                1  


In [7]:
!mkdir -p /content/datasets/mobile
!kaggle datasets download \
    -d girish17019/mobile-phone-defect-segmentation-dataset \
    -p /content/datasets/mobile

Dataset URL: https://www.kaggle.com/datasets/girish17019/mobile-phone-defect-segmentation-dataset
License(s): GNU Affero General Public License 3.0
100% 1.61G/1.61G [00:14<00:00, 123MB/s]



In [8]:
!unzip -q /content/datasets/mobile/*.zip \
    -d /content/datasets/mobile/extracted

In [9]:
import os

for root, dirs, files in os.walk("/content/datasets/mobile/extracted"):
    level = root.replace("/content/datasets/mobile/extracted", "").count(os.sep)

    if level < 2:
        print("  " * level + os.path.basename(root) + "/")

extracted/
  ground_truth_2/
  scratch/
  good/
  stain/
  oil/
  ground_truth_1/


In [10]:
!mkdir -p /content/datasets/mvtec

!kaggle datasets download \
    -d ipythonx/mvtec-ad \
    -p /content/datasets/mvtec

Dataset URL: https://www.kaggle.com/datasets/ipythonx/mvtec-ad
License(s): copyright-authors
100% 4.91G/4.91G [00:46<00:00, 113MB/s]



In [11]:
!unzip -q /content/datasets/mvtec/*.zip \
    -d /content/datasets/mvtec/extracted

In [12]:
import os

for item in os.listdir("/content/datasets/mvtec/extracted"):
    print(item)

readme.txt
zipper
leather
metal_nut
toothbrush
cable
tile
license.txt
hazelnut
wood
bottle
screw
transistor
carpet
pill
capsule
grid


In [13]:
import os
import cv2
import json
import math
import random
import sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from PIL import Image

from pathlib import Path
from collections import Counter, defaultdict

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms, models

import faiss

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", DEVICE)

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

BASE_DIR = Path("/content/InspectX_AI")

DATA_DIR = BASE_DIR / "data"
MODEL_DIR = BASE_DIR / "models"
OUTPUT_DIR = BASE_DIR / "outputs"
VECTOR_DIR = BASE_DIR / "vector_db"

for directory in [
    DATA_DIR,
    MODEL_DIR,
    OUTPUT_DIR,
    VECTOR_DIR
]:
    directory.mkdir(parents=True, exist_ok=True)

print(BASE_DIR)

Device: cuda
/content/InspectX_AI


In [14]:
MOBILE_ROOT = Path("/content/datasets/mobile/extracted")

def find_mobile_root(base):
    required = {"good", "oil", "scratch", "stain"}

    for root, dirs, files in os.walk(base):
        if required.issubset(set(dirs)):
            return Path(root)

    return None

MOBILE_ROOT = find_mobile_root(MOBILE_ROOT)

print("Mobile dataset:", MOBILE_ROOT)

if MOBILE_ROOT is None:
    raise FileNotFoundError(
        "Could not find good/oil/scratch/stain folders."
    )

Mobile dataset: /content/datasets/mobile/extracted


In [15]:
classes = ["good", "oil", "scratch", "stain"]

image_extensions = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".webp"
}

mobile_files = []

for cls in classes:

    class_dir = MOBILE_ROOT / cls

    files = [
        p for p in class_dir.rglob("*")
        if p.suffix.lower() in image_extensions
    ]

    print(f"{cls}: {len(files)}")

    for f in files:
        mobile_files.append({
            "path": str(f),
            "label": cls
        })

mobile_df = pd.DataFrame(mobile_files)

print("\nTotal:", len(mobile_df))
print(mobile_df["label"].value_counts())

good: 20
oil: 400
scratch: 400
stain: 400

Total: 1220
label
oil        400
scratch    400
stain      400
good        20
Name: count, dtype: int64
